# 립 자연스러움 채점: 비공개 Colab
LaPa 비상업 연구 전용. DB에서 export-colab으로 내보낸 v2 묶음만 사용합니다. 팀원 사진은 업로드하지 않습니다. L4 이상 권장. T4는 4비트와 fp16을 사용하며 bf16을 쓰지 않습니다.

In [ ]:
!nvidia-smi


## 1. 비공개 Drive 마운트 및 데이터 무결성 확인
본인만 접근 가능한 폴더에 묶음을 두고 ZIP SHA-256을 입력하세요. 작업이 끝나면 비공개 데이터와 Colab 임시 파일을 삭제하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, zipfile
DRIVE = Path('/content/drive/MyDrive/lip-rubric-private')
ZIP = DRIVE / 'colab_pack.zip'
EXPECTED_SHA256 = '여기에_담당자가_출력받은_SHA256_입력'
actual = hashlib.sha256(ZIP.read_bytes()).hexdigest()
assert actual.lower() == EXPECTED_SHA256.lower(), f'SHA-256 불일치: {actual}'
ROOT = Path('/content/lip-rubric-pack'); ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for name in z.namelist():
        assert (ROOT / name).resolve().is_relative_to(ROOT.resolve()), '잘못된 압축 경로'
    z.extractall(ROOT)
print('검증 완료:', actual)


## 2. 의존성 설치 및 환경 기록
설치에는 네트워크가 필요합니다. 모델은 외부 API가 아닌 Colab GPU에서 직접 실행합니다.

In [ ]:
%pip install 'transformers==4.57.6' 'accelerate==1.12.0' 'bitsandbytes==0.49.0' 'qwen-vl-utils==0.0.14'
import subprocess, sys
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / 'pip-freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'], text=True))
with (DRIVE / 'nvidia-smi.txt').open('w') as f: subprocess.run(['nvidia-smi'], stdout=f, check=True)


## 3. 모델 카드 확인 및 버전 기록
실행 전에 Hugging Face의 VLM과 임베딩 모델 카드에서 라이선스를 확인하고 정확한 커밋 리비전을 넣으세요. CLIP 라이선스는 현재 확인되지 않았습니다. 모델 ID도 필요하면 설정에서 변경할 수 있습니다.

In [ ]:
import json
cfg_path = ROOT / 'judges.json'
cfg = json.loads(cfg_path.read_text())
print('VLM:', cfg['vlm']['model_id'], cfg['vlm']['license'])
print('임베딩:', cfg['embedding']['model_id'], cfg['embedding']['license'])
cfg['vlm']['revision'] = '여기에_확인한_VLM_리비전'
cfg['embedding']['revision'] = '여기에_확인한_임베딩_리비전'
assert not any('여기에' in cfg[k]['revision'] for k in ('vlm','embedding')), '정확한 리비전을 입력하세요'
cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2))


## 4. 사례 검색 및 채점
입술 확대만 임베딩하고 같은 인물을 제외하여 pool 사례 3개를 고릅니다. A 제로샷, B RAG와 세 가지 사례 순서 반복, C 이웃 평균을 수행합니다. 각 결과는 Drive에 바로 기록되며 같은 설정으로 재실행하면 이어갑니다. 런타임 시간이 길 수 있습니다.

In [ ]:
result = DRIVE / 'judge_results.csv'
subprocess.run([sys.executable, str(ROOT/'judge.py'), '--pack', str(ROOT), '--output', str(result)], check=True)
print('저장 위치:', result)


## 5. 결과 내려받기
`judge_results.csv`를 담당자의 `research/data/lip_rubric/results/`에 저장하고 `db.py import-judge <CSV경로>`로 넣은 뒤 `analyze.py`를 실행합니다. 원본 이미지와 결과는 비공개로 보관하고 실험 종료 후 정해진 절차에 따라 삭제합니다.